# Gold Layer - Business-Level Aggregations
## Healthcare Lakehouse Project

This notebook creates the **Gold layer** with business-ready aggregated tables for analytics and reporting.

### Gold Tables Created:
1. **gold.top_drugs**: Drug-level cost and claims analysis
2. **gold.top_providers**: Provider-level performance metrics
3. **gold.state_summary**: State-level healthcare spending summary

### Key Metrics:
- Total cost, claims, and beneficiaries
- Cost per claim
- Cost per beneficiary
- Claims per beneficiary
- Unique drugs and providers

### Step 1: Import Required Libraries

In [ ]:
# Databricks notebook source
from pyspark.sql import functions as F
from delta.tables import DeltaTable

### Step 2: Load Utility Variables and Configure Widgets

In [ ]:
# MAGIC %run "/Workspace/Users/pawanvirat32@gmail.com/healthcare_lakehouse_project/01_setup/utilities"

In [ ]:
print(gold_schema)

dbutils.widgets.text('catalog', 'healthcare_lakehouse', 'Catalog')
dbutils.widgets.text('data_source', 'by_provider_drug', 'Data Source')

catalog = dbutils.widgets.get('catalog')
data_source = dbutils.widgets.get('data_source')

print(f"catalog: {catalog}")
print(f"data_source: {data_source}")

### Step 3: Load Silver Tables
Reads the fact and dimension tables from the Silver layer.

In [ ]:
fact_df = spark.read.table(f"{catalog}.{silver_schema}.fact_prescriptions")
drug_df = spark.read.table(f"{catalog}.{silver_schema}.drug")
provider_df = spark.read.table(f"{catalog}.{silver_schema}.provider")

In [ ]:
fact_df.show(10)

### Step 4: Join Fact with Dimension Tables
Creates a unified fact table with provider and drug IDs for analysis.

In [ ]:
fact_df = df_bronze.alias("f") \
    .join(provider_df.alias("p"), "prscrbr_npi") \
    .join(drug_df.alias("d"), "gnrc_name") \
    .select(
        F.col("p.provider_id"),
        F.col("d.drug_id"),
        F.col("f.prscrbr_state_abrvtn"),   # choose ONE
        F.col("f.tot_clms"),
        F.col("f.tot_drug_cst"),
        F.col("f.tot_benes")
    )

In [ ]:
fact_df.show(10)

### Step 5: Create Top Drugs Table
Aggregates metrics by drug to identify high-cost and high-volume drugs.

In [ ]:
gold_drug = fact_df.groupBy("drug_id") \
    .agg(
        F.sum("tot_drug_cst").alias("total_cost"),
        F.sum("tot_clms").alias("total_claims")
    )

gold_drug = gold_drug.withColumn(
    "cost_per_claim",
    F.col("total_cost") / F.col("total_claims")
)

gold_drug = gold_drug.join(
    df_drug.select("drug_id", "gnrc_name"),
    "drug_id"
)

In [ ]:
gold_drug.show(10)

In [ ]:
gold_drug.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(f"{catalog}.gold.top_drugs")

### Step 6: Create Top Providers Table
Aggregates metrics by provider to identify top prescribers.

In [ ]:
# Aggregate by provider
gold_provider = fact_df.groupBy("provider_id") \
    .agg(
        F.sum("tot_drug_cst").alias("total_cost"),
        F.sum("tot_clms").alias("total_claims"),
        F.sum("tot_benes").alias("total_beneficiaries"),
        F.countDistinct("drug_id").alias("unique_drugs")
    )

# Calculate cost per claim
gold_provider = gold_provider.withColumn(
    "cost_per_claim",
    F.col("total_cost") / F.col("total_claims")
)

In [ ]:
# Join with provider details
gold_provider = gold_provider.join(
    provider_df.select(
        "provider_id",
        "prscrbr_npi",
        "prscrbr_first_name",
        "prscrbr_last_org_name",
        "prscrbr_city",
        "prscrbr_state_abrvtn",
        "prscrbr_cntry"
    ),
    "provider_id"
)

In [ ]:
display(gold_provider.orderBy(F.desc("total_cost")).limit(20))

In [ ]:
gold_provider.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(f"{catalog}.gold.top_providers")

### Step 7: Create State Summary Table
Aggregates metrics by state for geographic analysis.

In [ ]:
# Aggregate by state
gold_state = fact_df.groupBy("prscrbr_state_abrvtn") \
    .agg(
        F.sum("tot_drug_cst").alias("total_cost"),
        F.sum("tot_clms").alias("total_claims"),
        F.sum("tot_benes").alias("total_beneficiaries"),
        F.countDistinct("drug_id").alias("unique_drugs"),
        F.countDistinct("provider_id").alias("unique_providers")
    )

# Calculate derived metrics
gold_state = gold_state.withColumn(
    "cost_per_claim",
    F.col("total_cost") / F.col("total_claims")
).withColumn(
    "cost_per_beneficiary",
    F.col("total_cost") / F.col("total_beneficiaries")
).withColumn(
    "claims_per_beneficiary",
    F.col("total_claims") / F.col("total_beneficiaries")
)

In [ ]:
# Preview top spending states
display(gold_state.orderBy(F.desc("total_cost")).limit(20))

In [ ]:
gold_state.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(f"{catalog}.gold.state_summary")